| Tool           | Job                       |
| -------------- | ------------------------- |
| Dropout        | don't rely on few neurons |
| BatchNorm      | stabilise learning        |
| Augmentation   | more varied data          |
| Early stopping | stop before overfitting   |


A good CNN should learn general visual rules, not memorise exact training photos. Dropout, BatchNorm, Augmentation, and Early Stopping are tools that force generalisation.

In [5]:
# import torch
# import torch.nn as nn
# import torchvision
# import torchvision.models as models

# # Load ResNet18 pretrained on ImageNet
# model = models.resnet18(weights='IMAGENET1K_V1')

# # Inspect the full architecture
# print(model)

# # How many parameters total?
# total = sum(p.numel() for p in model.parameters())
# print(f"\nTotal params: {total:,}")   # ~11 million

In [6]:
# # The last layer — outputs 1000 ImageNet class scores
# print(model.fc)
# # Linear(in_features=512, out_features=1000, bias=True)

# # Check input to the fc layer
# print("FC input features:", model.fc.in_features)   # 512


In [7]:
# # Swap the 1000-class head for a 10-class head
# model.fc = nn.Sequential(
#     nn.Linear(512, 256),
#     nn.ReLU(),
#     nn.Dropout(0.3),
#     nn.Linear(256, 10)
# )

# # Confirm the change
# print(model.fc)
# print("New total params:", sum(p.numel() for p in model.parameters()))


In [8]:
# # Freeze ALL layers first
# for param in model.parameters():
#     param.requires_grad = False

# # Unfreeze only the new head
# for param in model.fc.parameters():
#     param.requires_grad = True

# # Check: how many params are trainable?
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
# print(f"Trainable: {trainable:,}  |  Frozen: {frozen:,}")

Entire flow

This topic is:

Old CNN

Start from zero

Learn everything

Transfer learning

Use pretrained vision knowledge

Replace final head

Freeze backbone

Train only new classifier

| Step              | Meaning                  |
| ----------------- | ------------------------ |
| Load ResNet18     | pretrained vision model  |
| 11M params        | learned weights          |
| model.fc          | final classifier         |
| Replace fc        | switch 1000 → 10 classes |
| Freeze backbone   | preserve learned vision  |
| Train head only   | adapt to new task        |
| Transfer learning | reuse old knowledge      |


In [9]:
# import torchvision.transforms as transforms

# train_transform = transforms.Compose([
#     transforms.Resize(224),                        # ResNet18 expects 224x224
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomCrop(224, padding=8),
#     transforms.ToTensor(),
#     transforms.Normalize((0.485,0.456,0.406),     # ImageNet stats — use these
#                          (0.229,0.224,0.225))
# ])
# val_transform = transforms.Compose([
#     transforms.Resize(224),
#     transforms.ToTensor(),
#     transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
# ])

# train_dataset = torchvision.datasets.CIFAR10('./data', train=True,  transform=train_transform)
# val_dataset   = torchvision.datasets.CIFAR10('./data', train=False, transform=val_transform)
# train_loader  = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
# val_loader    = torch.utils.data.DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0)

Big takeaway

CNN from scratch:
Teach vision

Transfer learning:
Borrow vision

And real-world ML almost always chooses:
Borrow first

In [10]:
# import torch.optim as optim

# device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model     = model.to(device)
# criterion = nn.CrossEntropyLoss()

# # Only pass trainable params to optimiser
# optimizer = optim.Adam(
#     filter(lambda p: p.requires_grad, model.parameters()),
#     lr=0.001
# )

# def train_epoch(model, loader, optimizer, criterion, device):
#     model.train()
#     correct, total, total_loss = 0, 0, 0.0
#     for imgs, labels in loader:
#         imgs, labels = imgs.to(device), labels.to(device)
#         optimizer.zero_grad()
#         out  = model(imgs)
#         loss = criterion(out, labels)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#         _, preds = torch.max(out, 1)
#         correct  += (preds == labels).sum().item()
#         total    += labels.size(0)
#     return total_loss/len(loader), correct/total

# def eval_epoch(model, loader, criterion, device):
#     model.eval()
#     correct, total, total_loss = 0, 0, 0.0
#     with torch.no_grad():
#         for imgs, labels in loader:
#             imgs, labels = imgs.to(device), labels.to(device)
#             out  = model(imgs)
#             loss = criterion(out, labels)
#             total_loss += loss.item()
#             _, preds = torch.max(out, 1)
#             correct  += (preds == labels).sum().item()
#             total    += labels.size(0)
#     return total_loss/len(loader), correct/total

# print("=== Phase 1: Head only (frozen backbone) ===")
# for epoch in range(5):
#     tl, ta = train_epoch(model, train_loader, optimizer, criterion, device)
#     vl, va = eval_epoch(model, val_loader, criterion, device)
#     print(f"Epoch {epoch+1} | train: {ta:.3f} | val: {va:.3f}")

Frozen ResNet
      +
Train new classifier
      +
Learn CIFAR labels
      =
Strong accuracy fast


| Code                  | Meaning                    |
| --------------------- | -------------------------- |
| device                | CPU/GPU choice             |
| CrossEntropyLoss      | multi-class error          |
| Adam                  | smart gradient descent     |
| filter(requires_grad) | train only head            |
| train()               | learning mode              |
| zero_grad()           | clear old gradients        |
| model(imgs)           | forward pass               |
| loss.backward()       | compute gradients          |
| optimizer.step()      | update weights             |
| eval()                | test mode                  |
| no_grad()             | no learning during eval    |
| epoch                 | one full pass through data |


Phase 1 trains only the new decision layer while keeping ResNet's visual knowledge untouched.


In [11]:
# # Unfreeze ALL layers
# for param in model.parameters():
#     param.requires_grad = True

# # Use a much lower learning rate — the pretrained weights are good,
# # you just want to nudge them slightly, not overwrite them
# optimizer = optim.Adam([
#     {'params': model.fc.parameters(),  'lr': 1e-3},   # head: normal lr
#     {'params': [p for name, p in model.named_parameters()
#                 if 'fc' not in name],  'lr': 1e-4},   # backbone: 10x lower lr
# ])
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# best_val_acc = 0
# print("\n=== Phase 2: Full fine-tuning ===")
# for epoch in range(10):
#     tl, ta = train_epoch(model, train_loader, optimizer, criterion, device)
#     vl, va = eval_epoch(model, val_loader, criterion, device)
#     scheduler.step()
#     print(f"Epoch {epoch+1:2d} | train: {ta:.3f} | val: {va:.3f}")

#     if va > best_val_acc:
#         best_val_acc = va
#         torch.save(model.state_dict(), 'resnet18_cifar10.pth')

# print(f"\nBest val accuracy: {best_val_acc:.3f}")

Think of ResNet as a senior chef trained for years.

Phase 1

You told chef:

"Don't change your cooking style. Just learn this new menu."

Only menu (head) changed.

Phase 2

Now you say:

"Your basics are good, but slightly adjust your cooking for this restaurant."

Entire chef improves.

That is fine-tuning.

Training flow (whole picture)


Pretrained ResNet
        ↓
Freeze backbone
        ↓
Train head only
        ↓
Good CIFAR classifier
        ↓
Unfreeze all
        ↓
Low LR backbone
High LR head
        ↓
Fine-tune gently
        ↓
Scheduler lowers LR
        ↓
Save best model


Core intuition to remember

Transfer learning = don't start from zero.

Phase 1:

Learn new task quickly.

Phase 2:

Carefully adapt existing intelligence.

That is why ResNet jumps from ~65–70% (basic CNN) to ~85%+ often with much less training.